In [1]:
import json, os, random, glob
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# ─── Config ───────────────────────────────────────────────────────────────────
TRAIN_PATH   = "/kaggle/input/datasets/akgiiith/cp-project/psycomark_train.jsonl"
DEV_PATH     = "/kaggle/input/datasets/akgiiith/cp-project/psycomark_dev.jsonl"  
MODEL_NAME   = "microsoft/deberta-v3-large"
OUTPUT_DIR   = "/kaggle/working/deberta_psycomark"
MAX_LEN      = 256  # Paper optimal length
N_FOLDS      = 5
SEED         = 42
THRESHOLD    = 0.595 # Truth Gradient optimized decision boundary

# TRUTH GRADIENT MATCH: Binary Classification (Can't Tell -> Yes)
LABEL2ID = {"No": 0, "Yes": 1}
ID2LABEL = {0: "No", 1: "Yes"}

# ─── Reproducibility ──────────────────────────────────────────────────────────
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# ─── Data loading ─────────────────────────────────────────────────────────────
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    texts, labels = [], []
    for r in records:
        texts.append(r["text"])
        label_str = r["conspiracy"]
        
        # TRUTH GRADIENT MATCH: Remap "Can't tell" to "Yes"
        if label_str == "Can't tell":
            label_str = "Yes"
            
        labels.append(LABEL2ID[label_str])
    return texts, labels

# Load train + dev, combine them
train_records = load_jsonl(TRAIN_PATH)
dev_records   = load_jsonl(DEV_PATH)
all_records   = train_records + dev_records

texts, labels = extract_fields(all_records)
labels_np = np.array(labels)

print(f"Total samples: {len(texts)}")
for k, v in LABEL2ID.items():
    print(f"  {k}: {(labels_np == v).sum()}")

# ─── Dataset & Training Setup ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ConspDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            texts,
            max_length=MAX_LEN,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

# WEIGHTED CROSS ENTROPY CHECK: Calculating penalty weights
def compute_class_weights(labels_np):
    N, C = len(labels_np), len(LABEL2ID)
    weights = []
    for c in range(C):
        nc = (labels_np == c).sum()
        if nc == 0:
            weights.append(0.0) 
        else:
            weights.append(N / (C * nc))
    return torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        current_dtype = logits.dtype
        
        # COMBINED APPROACH: Weighted Cross Entropy + Truth Gradient Label Smoothing
        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(current_dtype),
            label_smoothing=0.10
        )
        
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1":    f1_score(labels, preds, average="macro"), # TRUTH GRADIENT MATCH
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

def make_training_args(fold_dir):
    return TrainingArguments(
        output_dir                  = fold_dir,
        num_train_epochs            = 7,
        per_device_train_batch_size = 4,         
        per_device_eval_batch_size  = 8,         
        gradient_accumulation_steps = 8,         
        learning_rate               = 3e-6,
        weight_decay                = 0.01,
        warmup_ratio                = 0.086,
        fp16                        = False,     
        gradient_checkpointing      = True,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "macro_f1", # TRUTH GRADIENT MATCH
        greater_is_better           = True,
        logging_steps               = 50,
        report_to                   = "none",
        seed                        = SEED,
        save_total_limit            = 1,         
    )

# ─── 5-Fold Cross-Validation ──────────────────────────────────────────────────
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
texts_np     = np.array(texts)
fold_models  = []          
oof_logits   = np.zeros((len(texts), len(LABEL2ID)))  

for fold, (tr_idx, val_idx) in enumerate(skf.split(texts_np, labels_np)):
    print(f"\n{'='*60}\n  FOLD {fold+1}/{N_FOLDS}\n{'='*60}")

    tr_texts,  val_texts  = texts_np[tr_idx].tolist(),  texts_np[val_idx].tolist()
    tr_labels, val_labels = labels_np[tr_idx].tolist(), labels_np[val_idx].tolist()

    tr_dataset  = ConspDataset(tr_texts,  tr_labels)
    val_dataset = ConspDataset(val_texts, val_labels)

    cw = compute_class_weights(np.array(tr_labels))
    print(f"  Class weights → No:{cw[0]:.3f}  Yes:{cw[1]:.3f}")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels  = len(LABEL2ID),
        id2label    = ID2LABEL,
        label2id    = LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    fold_dir = os.path.join(OUTPUT_DIR, f"fold_{fold+1}")
    trainer  = WeightedTrainer(
        class_weights   = cw,
        model           = model,
        args            = make_training_args(fold_dir),
        train_dataset   = tr_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=4)],
    )

    existing_checkpoints = glob.glob(os.path.join(fold_dir, "checkpoint-*"))
    if len(existing_checkpoints) > 0:
        print(f"  Found existing checkpoint. Resuming Fold {fold+1}...")
        trainer.train(resume_from_checkpoint=True)
    else:
        trainer.train()

    raw = trainer.predict(val_dataset)
    oof_logits[val_idx] = raw.predictions

    # NOTE: OOF report here uses standard 0.5 threshold temporarily just for logging fold progress
    val_preds = np.argmax(raw.predictions, axis=-1)
    print(f"\n  Fold {fold+1} OOF report:")
    print(classification_report(val_labels, val_preds,
                                labels=list(LABEL2ID.values()), 
                                target_names=list(LABEL2ID.keys()),
                                zero_division=0))               

    best_ckpt = trainer.state.best_model_checkpoint
    if best_ckpt is None and len(existing_checkpoints) > 0:
        best_ckpt = existing_checkpoints[0] 
        
    fold_models.append(best_ckpt)
    print(f"  Best checkpoint: {best_ckpt}")

# ─── Inference helper (soft-voting ensemble) ──────────────────────────────────
import gc

def predict_ensemble(texts_to_predict: list[str]) -> list[str]:
    dataset = ConspDataset(texts_to_predict, [0] * len(texts_to_predict))
    avg_probs = np.zeros((len(texts_to_predict), len(LABEL2ID)))

    for ckpt in fold_models:
        m = AutoModelForSequenceClassification.from_pretrained(ckpt).eval()
        tmp_trainer = Trainer(model=m)
        out = tmp_trainer.predict(dataset)
        
        logits = out.predictions
        probs  = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
        avg_probs += probs
        
        del m, tmp_trainer
        torch.cuda.empty_cache()
        gc.collect()

    avg_probs /= len(fold_models)
    
    # TRUTH GRADIENT MATCH: Use optimized 0.595 threshold
    pred_ids = (avg_probs[:, 1] >= THRESHOLD).astype(int)
    
    return [ID2LABEL[i] for i in pred_ids]

# ─── Quick smoke-test on dev set ─────────────────────────────────────────────
if __name__ == "__main__":
    print("\nRunning ensemble inference on dev set …")
    dev_texts, dev_labels_ids = extract_fields(dev_records)
    dev_preds = predict_ensemble(dev_texts)
    dev_pred_ids = [LABEL2ID[p] for p in dev_preds]

    y_t = np.array(dev_labels_ids)
    y_p = np.array(dev_pred_ids)

    print(f"\nDev set Macro F1 (with tau={THRESHOLD}): "
          f"{f1_score(y_t, y_p, average='macro'):.4f}")
    print(classification_report(y_t, y_p, target_names=["No", "Yes"]))

Total samples: 4416
  No: 2040
  Yes: 2376


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


  FOLD 1/5
  Class weights → No:1.082  Yes:0.929


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Dev set Macro F1 (with tau=0.595): 0.3333
              precision    recall  f1-score   support

          No       0.50      1.00      0.67        50
         Yes       0.00      0.00      0.00        50

    accuracy                           0.50       100
   macro avg       0.25      0.50      0.33       100
weighted avg       0.25      0.50      0.33       100



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
